# Notebook 003. Vector pre-processing
-------
Generates the analysis-ready vectors in `data/processed/vectors`.

1. **CORINE.** Clip the all-Romania GeoPackage to the AOI and extract the forest classes (311 / 312 / 313).
2. **Parcel map.** Clean the cadastral parcel map: reproject, repair, clip, de-overlap, drop non-productive units, standardise composition and age, assign parcel ids.
3. **Management plans.** Clean the seven plans and stack them into one layer.
4. **Virgin and quasi-virgin forests.** Clean the expert-verified polygons.
5. **Ownership.** Clean the ownership layer.

Steps 2 to 5 need the forest records compiled by Fundația Conservation Carpathia. If they are
not present, those cells skip themselves and the *published dataset* cell after the parcel-map
section rebuilds the parcel map and the reference labels from the Zenodo release instead, so
the notebook still runs top to bottom.

## CORINE Land Cover

Extracts the forest classes from the CORINE Land Cover 2018 inventory within the AOI. The all-Romania polygons are read for the AOI bounding box, reprojected to EPSG:3035, filtered to the broad-leaved (311), coniferous (312) and mixed (313) forest classes, and clipped to the AOI polygon. The result is stored as a polygon layer carrying the CORINE code and forest type. EPSG:3035 is an equal-area metric coordinate reference system, so reported areas are in true hectares.

In [ ]:
# CORINE Land Cover: extract the forest classes within the AOI. The all-Romania CLC2018
# polygons are read for the AOI bounding box, reprojected to EPSG:3035, filtered to the
# broad-leaved, coniferous and mixed forest classes (311/312/313), and clipped to the
# AOI polygon. Stored as a vector; rasterisation to the reference grid is done at
# consumption. EPSG:3035 is an equal-area metric CRS, so areas are in true hectares.
import logging

import geopandas
import pyogrio
from pyproj import CRS as PYPROJ_CRS

from utils import terminology, vector_io
from utils.paths import get_project_paths
from utils.vector_io import load_aoi, repair_geometries

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()

CORINE_RAW = paths.raw / "vectors" / "corine_land_cover" / "corine_clc2018_romania.gpkg"
CORINE_VEC_OUT = paths.processed / "vectors" / "corine_land_cover" / "corine_forest_aoi_3035.gpkg"
CORINE_VEC_OUT.parent.mkdir(parents=True, exist_ok=True)

# CLC level-3 code column varies by distribution; detect from these (case-insensitive).
CODE_CANDIDATES = ("code_18", "code18", "clc_code", "clc", "code")
AREA_CANDIDATES = ("area_ha", "shape_area")  # source area field, if present
FOREST_CODES = {str(k) for k in terminology.CORINE_FOREST_CLASSES}  # {'311','312','313'}

aoi = load_aoi(dissolve=True)  # EPSG:3035

if CORINE_VEC_OUT.exists():
    print(f"[skip] CORINE forest vector already present: {CORINE_VEC_OUT.name}")
else:
    # Resolve the single layer and its CRS without reading geometries.
    layers = geopandas.list_layers(CORINE_RAW)
    if len(layers) != 1:
        raise ValueError(f"Expected one layer; found {list(layers['name'])}.")
    layer = layers["name"].iloc[0]
    info = pyogrio.read_info(CORINE_RAW, layer=layer)
    src_crs = info["crs"]
    print(f"[audit] CORINE: layer '{layer}', {src_crs}, {info['features']:,} features (full file)")

    # Read only features within the AOI bounding box (in the file's CRS), then reproject.
    bbox = tuple(aoi.to_crs(src_crs).total_bounds)
    gdf = geopandas.read_file(CORINE_RAW, layer=layer, bbox=bbox)
    print(f"[bbox-read] {len(gdf):,} features within the AOI bounding box")

    if not PYPROJ_CRS.from_user_input(gdf.crs).equals(PYPROJ_CRS.from_user_input(terminology.CRS)):
        gdf = gdf.to_crs(terminology.CRS)
        print(f"[reproject] {src_crs} -> {terminology.CRS}")
    else:
        print(f"[reproject] none needed; already {terminology.CRS}")

    # Detect the CLC code column and keep only the forest classes.
    cols_lower = {c.lower(): c for c in gdf.columns}
    code_col = next((cols_lower[c] for c in CODE_CANDIDATES if c in cols_lower), None)
    if code_col is None:
        raise ValueError(
            f"No CLC code column among {CODE_CANDIDATES}; columns: {list(gdf.columns)}"
        )
    area_col = next((cols_lower[c] for c in AREA_CANDIDATES if c in cols_lower), None)

    codes = gdf[code_col].astype("string").str.strip()
    forest = gdf.loc[codes.isin(FOREST_CODES)].copy()
    forest["clc_code"] = forest[code_col].astype("string").str.strip().astype(int)
    forest["forest_type"] = forest["clc_code"].map(dict(terminology.CORINE_FOREST_CLASSES))
    # Source CLC polygon area (unclipped), carried through to quantify boundary clip loss.
    if area_col is not None:
        forest["area_ha_clc"] = forest[area_col].astype("float64")
        keep = ["clc_code", "forest_type", "area_ha_clc", "geometry"]
    else:
        keep = ["clc_code", "forest_type", "geometry"]
    forest = forest[keep]
    print(
        f"[filter] code column '{code_col}'"
        + (f", area column '{area_col}'" if area_col else ", no source area column")
        + f"; {len(forest):,} forest polygons (311/312/313)"
    )

    # Clip to the AOI polygon and repair.
    forest = geopandas.clip(forest, aoi)
    forest = repair_geometries(forest)
    forest = forest[forest.geom_type.isin(("Polygon", "MultiPolygon"))].copy()

    # Clipped area is recomputed from geometry; source area (if present) is the unclipped
    # CLC figure, so the two differ only by what the AOI boundary trims.
    forest["area_ha"] = forest.geometry.area / 1e4
    agg = {"polygons": ("forest_type", "size"), "area_ha_clipped": ("area_ha", "sum")}
    if "area_ha_clc" in forest.columns:
        agg["area_ha_source"] = ("area_ha_clc", "sum")
    summary = forest.groupby("forest_type").agg(**agg).round(1)
    print("[clip] forest within the AOI polygon (written to vector):")
    print(summary.to_string())

    forest.to_file(CORINE_VEC_OUT, driver="GPKG")
    print(f"[write] {CORINE_VEC_OUT.name}: {len(forest):,} features")

print(f"[verify] {vector_io.audit_vector(CORINE_VEC_OUT)}")

## Parcel map

Clean the cadastral parcel map: reproject, repair, clip to the AOI, de-overlap, drop non-productive units, standardise composition and age, retain the virgin-forest record, and assign a unique parcel id.

In [ ]:
import geopandas as gpd

from utils.forestry import normalise_stand_age, standardise_composition
from utils.geometry_ops import clip_to_aoi_preserve_rows, deoverlap_polygons
from utils.paths import get_project_paths
from utils.terminology import CRS
from utils.vector_io import load_aoi, repair_geometries

# Source configuration: confirm these match the raw layer.
COMPOSITION_COL = "compoz"
AGE_COL = "varsta"
PV_COL = "PV"  # parcel-map virgin-forest record (Padure Virgina)
NONFOREST_COL = "UA"  # UA == "00" marks non-productive units (water, clearcuts, etc.)
NONFOREST_CODE = "00"
PLAN_YEAR = 2018  # cadastral baseline year for the parcel map

paths = get_project_paths()
raw_path = paths.raw / "vectors" / "forest_records" / "parcel_map_original.gpkg"
out_dir = paths.processed / "vectors" / "forest_records"
out_path = out_dir / "parcel_map_clean.gpkg"

if out_path.exists():
    print(f"[skip] parcel map already processed: {out_path}")
elif not raw_path.exists():
    print(
        f"[skip] forest records not available ({raw_path.name} missing); "
        "the published-dataset cell below rebuilds the parcel map instead"
    )
else:
    parcels = gpd.read_file(raw_path)
    n_raw = len(parcels)
    print(f"[load] {n_raw} parcels from {raw_path.name} ({parcels.crs})")

    parcels = parcels.to_crs(CRS)
    print(f"[reproject] -> {CRS}")

    before = len(parcels)
    n_with_z = int(parcels.geometry.has_z.sum())
    parcels = repair_geometries(parcels)
    print(
        f"[repair] {before} -> {len(parcels)} ({before - len(parcels)} empty dropped, "
        f"{n_with_z} flattened to 2D)"
    )

    before = len(parcels)
    parcels = clip_to_aoi_preserve_rows(parcels, load_aoi())
    print(f"[clip] {before} -> {len(parcels)} ({before - len(parcels)} outside AOI dropped)")

    parcels, stats = deoverlap_polygons(parcels, min_residual_fraction=0.40, return_stats=True)
    print(f"[deoverlap] {stats}")

    if NONFOREST_COL in parcels.columns:
        before = len(parcels)
        parcels = parcels[parcels[NONFOREST_COL].astype(str) != NONFOREST_CODE].copy()
        print(
            f"[strip] {before} -> {len(parcels)} ({before - len(parcels)} non-productive dropped; "
            f"{NONFOREST_COL} == '{NONFOREST_CODE}')"
        )

    species = parcels[COMPOSITION_COL].map(standardise_composition)
    age = parcels[AGE_COL].map(normalise_stand_age)
    pv = parcels[PV_COL].mask(parcels[PV_COL] == "", None)
    n_clearcut = int((species == "CLEARCUT").sum())
    n_valid = int(species.notna().sum()) - n_clearcut
    print(
        f"[clean] composition: {n_valid} valid, {n_clearcut} clearcut, "
        f"{int(species.isna().sum())} blank; age: {int(age.notna().sum())} present, "
        f"{int(age.isna().sum())} missing"
    )
    n_confirmed = int((pv == "Da").sum())
    n_flagged = int((pv.notna() & (pv != "Da")).sum())
    print(
        f"[pv] {n_confirmed} confirmed virgin (Da), {n_flagged} flagged, "
        f"{int(pv.isna().sum())} blank"
    )

    parcels = gpd.GeoDataFrame(
        {
            "parcel_id": [f"{i:05d}" for i in range(1, len(parcels) + 1)],
            "species_composition": species.to_numpy(),
            "average_stand_age": age.to_numpy(),
            "management_plan_year": PLAN_YEAR,
            "padure_virgina_pv": pv.to_numpy(),
        },
        geometry=parcels.geometry.to_numpy(),
        crs=parcels.crs,
    )
    out_dir.mkdir(parents=True, exist_ok=True)
    parcels.to_file(out_path, layer="parcel_map", driver="GPKG")
    print(f"[write] {n_raw} raw -> {len(parcels)} clean parcels at {out_path.name}")

### From the published dataset (no forest records)

If the forest records are not available, this cell rebuilds `parcel_map_clean.gpkg` and the
reference labels (`ogf_reference_labels.gpkg`, normally written by notebook 005) from the
published dataset [`ogf_labels_predictions.gpkg`](https://doi.org/10.5281/zenodo.22693148).
Put the file at `data/processed/vectors/labels/ogf_labels_predictions.gpkg`. Parcel
geometries, ids, labels and CORINE forest types are the study's own; roads and 1985-2020
disturbance are trimmed from the old-growth parcels exactly as notebook 005 does. Attributes
that exist only in the forest records (species composition, stand age) are left
empty. Existing layers are kept unless `FORCE = True`.

In [ ]:
import logging

from utils.paths import get_project_paths
from utils.published import published_labels_path, rebuild_from_published

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

FORCE = False  # rewrite the parcel map and labels even if they already exist

paths = get_project_paths()
published = published_labels_path(paths)
if published.exists():
    parcel_map_path, labels_path = rebuild_from_published(paths, force=FORCE)
    print(f"[rebuild] {parcel_map_path.name} and {labels_path.name} written from {published.name}")
else:
    print(f"[skip] no published dataset at {published}; the forest records are used instead")

## Management plans

Clean the seven management plans and stack them into one layer, each parcel carrying its standardised species composition, average stand age, plan year, and whether it comes from a private or public plan.

In [ ]:
import geopandas as gpd
import pandas as pd

from utils.forestry import normalise_stand_age, standardise_composition
from utils.geometry_ops import clip_to_aoi_preserve_rows, deoverlap_polygons
from utils.paths import get_project_paths
from utils.terminology import CRS
from utils.vector_io import load_aoi, repair_geometries

# Per-plan config: (file stem, composition column, year column, layer or None for the stem).
# Age uses AGE_COL throughout; privat/public is inferred from the stem. Confirm against raw files.
AGE_COL = "ta"
NONFOREST_COL = "UA"
NONFOREST_CODE = "00"
PLANS = [
    ("private_forests__as_privat_2015_2025v6", "compoz_act", "An_Amenaja", None),
    ("state_owned_forest__as_privat_2022_2024", "compoz_act", "An_Amenaja", None),
    ("state_owned_forest__as_privat_2019_2022", "compoz_act", "An_Amenaja", None),
    ("state_owned_forest__as_privat_2013_2018", "compoz_act", "An_Amenaja", None),
    ("state_owned_forest__as_public_2013_2018", "compoz", "an", "remaining_fields"),
    ("state_owned_forest__as_public_2019_2022", "compoz", "an", None),
    ("state_owned_forest__as_public_2022_2023", "compoz", "an", None),
]

paths = get_project_paths()
raw_dir = paths.raw / "vectors" / "forest_records"
out_dir = paths.processed / "vectors" / "forest_records"
out_path = out_dir / "management_plans_clean.gpkg"

missing_plans = [stem for stem, *_ in PLANS if not (raw_dir / f"{stem}.gpkg").exists()]
if out_path.exists():
    print(f"[skip] management plans already processed: {out_path}")
elif missing_plans:
    print(f"[skip] forest records not available ({len(missing_plans)} management plans missing)")
else:
    aoi = load_aoi()
    cleaned = []
    # De-overlap each plan internally; overlaps between plans are retained for later resolution.
    for stem, comp_col, year_col, layer_override in PLANS:
        layer = layer_override or stem
        private = True if "as_privat" in stem else False if "as_public" in stem else None
        print(f"\n[{stem}]")

        gdf = gpd.read_file(raw_dir / f"{stem}.gpkg", layer=layer)
        print(f"[load] {len(gdf)} parcels ({gdf.crs})")

        gdf = gdf.to_crs(CRS)

        before = len(gdf)
        n_with_z = int(gdf.geometry.has_z.sum())
        gdf = repair_geometries(gdf)
        print(
            f"[repair] {before} -> {len(gdf)} ({before - len(gdf)} empty dropped, "
            f"{n_with_z} flattened to 2D)"
        )

        before = len(gdf)
        gdf = clip_to_aoi_preserve_rows(gdf, aoi)
        print(f"[clip] {before} -> {len(gdf)} ({before - len(gdf)} outside AOI dropped)")

        gdf, stats = deoverlap_polygons(gdf, return_stats=True)
        print(f"[deoverlap] {stats}")

        if NONFOREST_COL in gdf.columns:
            before = len(gdf)
            gdf = gdf[gdf[NONFOREST_COL].astype(str) != NONFOREST_CODE].copy()
            print(
                f"[strip] {before} -> {len(gdf)} ({before - len(gdf)} non-productive dropped; "
                f"{NONFOREST_COL} == '{NONFOREST_CODE}')"
            )

        gdf = gdf.reset_index(drop=True)
        species = gdf[comp_col].map(standardise_composition)
        age = gdf[AGE_COL].map(normalise_stand_age)
        n_clearcut = int((species == "CLEARCUT").sum())
        n_valid = int(species.notna().sum()) - n_clearcut
        print(
            f"[clean] composition: {n_valid} valid, {n_clearcut} clearcut, "
            f"{int(species.isna().sum())} blank; age: {int(age.notna().sum())} present, "
            f"{int(age.isna().sum())} missing"
        )

        plan_clean = gpd.GeoDataFrame(
            {
                "species_composition": species,
                "average_stand_age": age,
                "management_plan_year": pd.to_numeric(gdf[year_col], errors="coerce").astype(
                    "Int64"
                ),
            },
            geometry=gdf.geometry,
            crs=gdf.crs,
        )
        plan_clean["private"] = private
        cleaned.append(plan_clean)

    plans = gpd.GeoDataFrame(pd.concat(cleaned, ignore_index=True), geometry="geometry", crs=CRS)
    plans["private"] = plans["private"].astype("boolean")
    n_clearcut = int((plans["species_composition"] == "CLEARCUT").sum())
    n_valid = int(plans["species_composition"].notna().sum()) - n_clearcut
    n_blank = int(plans["species_composition"].isna().sum())
    n_aged = int(plans["average_stand_age"].notna().sum())
    n_privat = int(plans["private"].sum())
    print(
        f"\n[combined] {len(plans)} parcels from {len(PLANS)} plans "
        f"({n_privat} privat, {len(plans) - n_privat} public)"
    )
    print(
        f"[clean] composition {n_valid} valid, {n_clearcut} clearcut, {n_blank} blank; "
        f"age {n_aged} present"
    )
    yrs = plans["management_plan_year"].dropna()
    if len(yrs):
        print(f"[year] {len(yrs)} dated, range {int(yrs.min())}-{int(yrs.max())}")

    out_dir.mkdir(parents=True, exist_ok=True)
    plans.to_file(out_path, layer="management_plans", driver="GPKG")
    print(f"[write] {len(plans)} parcels -> {out_path.name}")

## Virgin and quasi-virgin forests

Clean the FCC-verified virgin and quasi-virgin forest polygons: reproject, repair, clip to the AOI, and de-overlap into a disjoint layer.

In [ ]:
import geopandas as gpd

from utils.geometry_ops import clip_to_aoi_preserve_rows, deoverlap_polygons
from utils.paths import get_project_paths
from utils.terminology import CRS
from utils.vector_io import load_aoi, repair_geometries

RAW_FILE = "FCC_confirmed_virgin_semi_virgin_forests.gpkg"
RAW_LAYER = "FCC_confirmed_primary_forests"

paths = get_project_paths()
raw_path = paths.raw / "vectors" / "forest_records" / RAW_FILE
out_dir = paths.processed / "vectors" / "forest_records"
out_path = out_dir / "virgin_forests_clean.gpkg"

if out_path.exists():
    print(f"[skip] virgin forests already processed: {out_path}")
elif not raw_path.exists():
    print(f"[skip] forest records not available ({raw_path.name} missing)")
else:
    virgin = gpd.read_file(raw_path, layer=RAW_LAYER)
    print(f"[load] {len(virgin)} polygons ({virgin.crs})")

    virgin = virgin.to_crs(CRS)

    before = len(virgin)
    n_with_z = int(virgin.geometry.has_z.sum())
    virgin = repair_geometries(virgin)
    print(
        f"[repair] {before} -> {len(virgin)} ({before - len(virgin)} empty dropped, "
        f"{n_with_z} flattened to 2D)"
    )

    before = len(virgin)
    virgin = clip_to_aoi_preserve_rows(virgin, load_aoi())
    print(f"[clip] {before} -> {len(virgin)} ({before - len(virgin)} outside AOI dropped)")

    virgin, stats = deoverlap_polygons(virgin, return_stats=True)
    print(f"[deoverlap] {stats}")

    virgin = gpd.GeoDataFrame(geometry=virgin.geometry.to_numpy(), crs=virgin.crs)
    area_ha = virgin.geometry.area.sum() / 10_000
    out_dir.mkdir(parents=True, exist_ok=True)
    virgin.to_file(out_path, layer="virgin_forests", driver="GPKG")
    print(f"[write] {len(virgin)} polygons, {area_ha:,.0f} ha -> {out_path.name}")

## Ownership

Clean the forest ownership layer: reproject, repair, clip to the AOI, de-overlap, and retain the ownership type.

In [ ]:
import geopandas as gpd

from utils.geometry_ops import clip_to_aoi_preserve_rows, deoverlap_polygons
from utils.paths import get_project_paths
from utils.terminology import CRS
from utils.vector_io import load_aoi, repair_geometries

RAW_FILE = "fagaras_mountains_ownership_type.gpkg"
RAW_LAYER = "ro_sci_122_fagaras_mountains_proprietari_final"
OWNERSHIP_COL = "Tip_propri"  # ownership type (Tip proprietate)

paths = get_project_paths()
raw_path = paths.raw / "vectors" / "forest_records" / RAW_FILE
out_dir = paths.processed / "vectors" / "forest_records"
out_path = out_dir / "ownership_type_clean.gpkg"

if out_path.exists():
    print(f"[skip] ownership already processed: {out_path}")
elif not raw_path.exists():
    print(f"[skip] forest records not available ({raw_path.name} missing)")
else:
    owners = gpd.read_file(raw_path, layer=RAW_LAYER)
    print(f"[load] {len(owners)} polygons ({owners.crs})")

    owners = owners.to_crs(CRS)

    before = len(owners)
    n_with_z = int(owners.geometry.has_z.sum())
    owners = repair_geometries(owners)
    print(
        f"[repair] {before} -> {len(owners)} ({before - len(owners)} empty dropped, "
        f"{n_with_z} flattened to 2D)"
    )

    before = len(owners)
    owners = clip_to_aoi_preserve_rows(owners, load_aoi())
    print(f"[clip] {before} -> {len(owners)} ({before - len(owners)} outside AOI dropped)")

    owners, stats = deoverlap_polygons(owners, return_stats=True)
    print(f"[deoverlap] {stats}")

    ownership = owners[OWNERSHIP_COL].astype("string").str.strip()
    ownership = ownership.mask(ownership == "", None)
    counts = ownership.value_counts(dropna=False)
    print("[type] " + ", ".join(f"{t}: {n}" for t, n in counts.items()))

    owners = gpd.GeoDataFrame(
        {"ownership_type": ownership.to_numpy()},
        geometry=owners.geometry.to_numpy(),
        crs=owners.crs,
    )
    area_ha = owners.geometry.area.sum() / 10_000
    out_dir.mkdir(parents=True, exist_ok=True)
    owners.to_file(out_path, layer="ownership_type", driver="GPKG")
    print(f"[write] {len(owners)} polygons, {area_ha:,.0f} ha -> {out_path.name}")